# Title: Car Diagnostics Audio Classification
### Multi-class classification of car sounds using 5 different ML approaches


# Abstract

# Introduction

# Problem Statement

# Motivation

# Data Source

1. Kaggle Car Diagnostics Dataset - https://www.kaggle.com/datasets/malakragaie/car-diagnostics-dataset

**Dataset structure:**
```
data/
├── braking_state/
│   ├── normal_brakes/
│   └── worn_out_brakes/
├── idle_state/
│   ├── low_oil/
│   ├── normal_engine_idle/
│   ├── power_steering/
│   └── serpentine_belt/
└── startup_state/
    ├── bad_ignition/
    ├── dead_battery/
    └── normal_engine_startup/
```

# Methodology Overview

Talk about these: XGBoost as baseline, CNN from scratch, CNN + LSTM, YAMNet Transfer Learning and PANNs Transfer Learning.

**Models compared:**
1. 🌲 MFCCs + XGBoost (baseline)
2. 🧠 CNN on Mel Spectrograms
3. 🔄 CNN-LSTM Hybrid
4. 🔁 Transfer Learning (YAMNet fine-tuned)
5. Transfer Learning (PANNs)


# Methodology Details

## Setup & Dependencies

In [1]:
# Install required packages (run once)
# !pip install librosa xgboost tensorflow tensorflow-hub scikit-learn matplotlib seaborn pandas numpy tqdm

import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from collections import Counter

import librosa
import librosa.display

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

import xgboost as xgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
import tensorflow_hub as hub

import importlib
import car_diagnostics as cd

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print(f'TensorFlow version: {tf.__version__}')
print(f'Librosa version: {librosa.__version__}')
print('All imports successful')

I0000 00:00:1780963146.693979  263048 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.21.0
Librosa version: 0.11.0
All imports successful


## Configuration

In [2]:
# EDIT THIS PATH
DATA_ROOT = Path('./data')   # ← Point this to your dataset root folder

# Audio config
SR         = 22050   # Sample rate to resample to
DURATION   = None    # None = use full file; set e.g. 3.0 to truncate/pad
N_MELS     = 128     # Mel bands for spectrograms
N_MFCC     = 40      # MFCC coefficients
HOP_LENGTH = 512
N_FFT      = 2048
FMAX       = 8000

# Training config
TEST_SIZE  = 0.15
VAL_SIZE   = 0.15
BATCH_SIZE = 32
EPOCHS     = 50
PATIENCE   = 10      # Early stopping patience

# Class map — parent state for each label (used for hierarchical visualisation)
STATE_MAP = {
    'normal_brakes':        'braking_state',
    'worn_out_brakes':      'braking_state',
    'low_oil':              'idle_state',
    'normal_engine_idle':   'idle_state',
    'power_steering':       'idle_state',
    'serpentine_belt':      'idle_state',
    'bad_ignition':         'startup_state',
    'dead_battery':         'startup_state',
    'normal_engine_startup':'startup_state',
}

STATE_COLORS = {
    'braking_state':  '#E07B54',
    'idle_state':     '#5B8DB8',
    'startup_state':  '#6BAE75',
}

print('Configuration set.')

Configuration set.


## Data Loading & Exploration

In [3]:
df = cd.discover_files(DATA_ROOT)
print(f'Total files found: {len(df)}')
print(f'Classes ({df["label"].nunique()}): {sorted(df["label"].unique())}')
print()
print(df.groupby(['state', 'label']).size().rename('count').to_string())

Total files found: 949
Classes (9): ['bad_ignition', 'dead_battery', 'low_oil', 'normal_brakes', 'normal_engine_idle', 'normal_engine_startup', 'power_steering', 'serpentine_belt', 'worn_out_brakes']

state          label                
braking_state  normal_brakes             77
               worn_out_brakes           76
idle_state     low_oil                  107
               normal_engine_idle       264
               power_steering           129
               serpentine_belt          116
startup_state  bad_ignition              62
               dead_battery              57
               normal_engine_startup     61


In [ ]:
cd.plot_distribution(df, STATE_COLORS)

## Audio Visualisations — Waveforms & Mel Spectrograms
### Waveforms

In [5]:
# Sample one file per class
samples = df.groupby('label').first().reset_index()
n_classes = len(samples)

print(f'Plotting waveforms and spectrograms for {n_classes} classes...')

Plotting waveforms and spectrograms for 9 classes...


In [ ]:
# Waveforms (1 per class)
cd.plot_waveforms(samples, SR, STATE_COLORS)

### Mel Spectograms

In [ ]:
# Mel Spectrograms (1 per class)
cd.plot_mel_spectograms(samples, SR, N_MELS, N_FFT, HOP_LENGTH, FMAX, STATE_COLORS)

### Side-by-side on Two Contrasting Classes

In [ ]:
# Side-by-side deep-dive on two contrasting classes
compare_labels = ['dead_battery', 'normal_engine_startup']
compare_rows   = df[df['label'].isin(compare_labels)].groupby('label').first()

cd.plot_comparisons(compare_rows, SR, N_MELS, N_FFT, HOP_LENGTH, FMAX, N_MFCC, STATE_COLORS)

## Data Augmentation

In [9]:
# Classes to augment and target multiplier
# 'normal_engine_idle' is excluded — it already has sufficient data.
# All other classes are doubled (multiplier=2 means 1 extra synthetic copy per file).

AUGMENT_CONFIG = {
    # braking_state
    'normal_brakes':         3,
    'worn_out_brakes':       3,
    # idle_state  (normal_engine_idle intentionally omitted)
    'low_oil':               2,
    'power_steering':        2,
    'serpentine_belt':       2,
    # startup_state
    'bad_ignition':          3,
    'dead_battery':          3,
    'normal_engine_startup': 3,
}

# Show before/after expected counts
print(f'{"Class":<25} {"Original":>10} {"Augmented":>10} {"Total":>10}')
print('─' * 58)
for label, multiplier in AUGMENT_CONFIG.items():
    orig = (df['label'] == label).sum()
    aug  = orig * (multiplier - 1)
    print(f'{label:<25} {orig:>10} {aug:>10} {orig+aug:>10}')

# Normal engine idle (not augmented)
label = 'normal_engine_idle'
orig  = (df['label'] == label).sum()
print(f'{label:<25} {orig:>10} {"(skipped)":>10} {orig:>10}')

Class                       Original  Augmented      Total
──────────────────────────────────────────────────────────
normal_brakes                     77        154        231
worn_out_brakes                   76        152        228
low_oil                          107        107        214
power_steering                   129        129        258
serpentine_belt                  116        116        232
bad_ignition                      62        124        186
dead_battery                      57        114        171
normal_engine_startup             61        122        183
normal_engine_idle               264  (skipped)        264


In [10]:
import soundfile as sf
import tempfile, uuid

import car_diagnostics as cd
importlib.reload(cd)

# Generate augmented audio files and extend df
# Augmented files are written to a temp directory as .wav files so that
# the rest of the pipeline (PANNs embedding extraction) can treat them
# identically to the originals.

AUG_DIR = Path(tempfile.mkdtemp(prefix='aug_'))
print(f'Augmented files will be written to: {AUG_DIR}\n')

aug_records = []

for label, multiplier in AUGMENT_CONFIG.items():
    source_rows = df[df['label'] == label]
    state       = source_rows.iloc[0]['state']
    n_copies    = multiplier - 1   # extra copies per original file

    print(f'{label} ({len(source_rows)} originals × {n_copies} copy = '
          f'{len(source_rows) * n_copies} new files)')

    for _, row in source_rows.iterrows():
        y, sr = librosa.load(row['path'], sr=SR, mono=True)

        for copy_idx in range(n_copies):
            out_name = f'{label}__{Path(row["path"]).stem}__aug{copy_idx}.wav'
            out_path = AUG_DIR / out_name
            if os.path.exists(out_path):
                continue
            y_aug  = cd.augment_waveform(y, sr, n_transforms=2)
            sf.write(str(out_path), y_aug, sr)
            aug_records.append({
                'path':  str(out_path),
                'label': label,
                'state': state,
            })

df_aug  = pd.DataFrame(aug_records)
df      = pd.concat([df, df_aug], ignore_index=True)

print(f'\nOriginal files : {len(df) - len(df_aug)}')
print(f'Augmented files: {len(df_aug)}')
print(f'Total          : {len(df)}')
print()
print(df.groupby(['state', 'label']).size().rename('count').to_string())

Augmented files will be written to: /tmp/aug_wvq_1jb4

normal_brakes (77 originals × 2 copy = 154 new files)
worn_out_brakes (76 originals × 2 copy = 152 new files)
low_oil (107 originals × 1 copy = 107 new files)
power_steering (129 originals × 1 copy = 129 new files)
serpentine_belt (116 originals × 1 copy = 116 new files)
bad_ignition (62 originals × 2 copy = 124 new files)
dead_battery (57 originals × 2 copy = 114 new files)
normal_engine_startup (61 originals × 2 copy = 122 new files)

Original files : 949
Augmented files: 1018
Total          : 1967

state          label                
braking_state  normal_brakes            231
               worn_out_brakes          228
idle_state     low_oil                  214
               normal_engine_idle       264
               power_steering           258
               serpentine_belt          232
startup_state  bad_ignition             186
               dead_battery             171
               normal_engine_startup    183


In [ ]:
cd.plot_distribution(df,STATE_COLORS)

## Feature Extraction

In [12]:
# Determine mel spectrogram time dimension from first file
sample_y = cd.load_audio(df.iloc[0]['path'], SR)
sample_mel = librosa.feature.melspectrogram(
    y=sample_y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
)
MEL_TIME_DIM = sample_mel.shape[1]
print(f'Mel spectrogram shape: ({N_MELS}, {MEL_TIME_DIM})')

Mel spectrogram shape: (128, 65)


In [13]:
# Extract all features (this may take a few minutes)
print('Extracting MFCC features...')
mfcc_features, mel_arrays, labels_raw = [], [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    mfcc_features.append(cd.extract_mfcc_features(row['path']))
    mel_arrays.append(cd.extract_mel_spectrogram_array(row['path'], target_length=MEL_TIME_DIM))
    labels_raw.append(row['label'])

X_mfcc = np.array(mfcc_features)
X_mel  = np.array(mel_arrays)        # shape: (N, N_MELS, MEL_TIME_DIM)
y_raw  = np.array(labels_raw)

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
n_classes = len(le.classes_)

print(f'\nMFCC feature matrix: {X_mfcc.shape}')
print(f'Mel spectrogram array: {X_mel.shape}')
print(f'Classes: {list(le.classes_)}')

Extracting MFCC features...


100%|██████████████████████████████████████████████████████████████████████████| 1967/1967 [00:41<00:00, 47.53it/s]



MFCC feature matrix: (1967, 268)
Mel spectrogram array: (1967, 128, 65)
Classes: [np.str_('bad_ignition'), np.str_('dead_battery'), np.str_('low_oil'), np.str_('normal_brakes'), np.str_('normal_engine_idle'), np.str_('normal_engine_startup'), np.str_('power_steering'), np.str_('serpentine_belt'), np.str_('worn_out_brakes')]


In [14]:
# Train / Val / Test splits
# First split off test set
idx = np.arange(len(y_enc))
idx_trainval, idx_test = train_test_split(
    idx, test_size=TEST_SIZE, random_state=SEED, stratify=y_enc
)
# Then split trainval into train and val
idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=SEED,
    stratify=y_enc[idx_trainval]
)

print(f'Train: {len(idx_train)}, Val: {len(idx_val)}, Test: {len(idx_test)}')

# MFCC splits
X_mfcc_train, X_mfcc_val, X_mfcc_test = X_mfcc[idx_train], X_mfcc[idx_val], X_mfcc[idx_test]

# Standardise MFCCs for XGBoost/classical models
scaler = StandardScaler()
X_mfcc_train_s = scaler.fit_transform(X_mfcc_train)
X_mfcc_val_s   = scaler.transform(X_mfcc_val)
X_mfcc_test_s  = scaler.transform(X_mfcc_test)

# Label splits
y_train, y_val, y_test = y_enc[idx_train], y_enc[idx_val], y_enc[idx_test]

# Mel spectrogram splits
X_mel_train = X_mel[idx_train][..., np.newaxis]   # add channel dim → (N, 128, T, 1)
X_mel_val   = X_mel[idx_val][..., np.newaxis]
X_mel_test  = X_mel[idx_test][..., np.newaxis]

print('Data splits are ready.')

Train: 1376, Val: 295, Test: 296
Data splits are ready.


## Model 1 — MFCCs + XGBoost (Baseline)

In [15]:
from sklearn.ensemble import RandomForestClassifier

print('Training XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1,
    early_stopping_rounds=20,
)
xgb_model.fit(
    X_mfcc_train_s, y_train,
    eval_set=[(X_mfcc_val_s, y_val)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_mfcc_test_s)
acc_xgb    = accuracy_score(y_test, y_pred_xgb)
f1_xgb     = f1_score(y_test, y_pred_xgb, average='weighted')

print(f'XGBoost  →  Accuracy: {acc_xgb:.4f}  |  Weighted F1: {f1_xgb:.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

Training XGBoost...
XGBoost  →  Accuracy: 0.8851  |  Weighted F1: 0.8827

                       precision    recall  f1-score   support

         bad_ignition       0.95      0.75      0.84        28
         dead_battery       0.86      0.96      0.91        26
              low_oil       0.74      0.62      0.68        32
        normal_brakes       0.85      0.97      0.91        35
   normal_engine_idle       0.97      0.95      0.96        40
normal_engine_startup       0.87      0.96      0.91        27
       power_steering       0.86      0.92      0.89        39
      serpentine_belt       0.94      0.91      0.93        35
      worn_out_brakes       0.91      0.88      0.90        34

             accuracy                           0.89       296
            macro avg       0.88      0.88      0.88       296
         weighted avg       0.89      0.89      0.88       296



In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cd.plot_confusion_matrix(y_test, y_pred_xgb, le.classes_,
                      'Confusion Matrix — XGBoost (MFCCs)', ax)
plt.tight_layout()
plt.show()

In [ ]:
# ── XGBoost Feature Importance Plots ─────────────────────────────────────────

# Feature names: match the order in extract_mfcc_features()
mfcc_idx    = range(N_MFCC)
feature_names = (
    [f'MFCC_mean_{i}'        for i in mfcc_idx] +
    [f'MFCC_std_{i}'         for i in mfcc_idx] +
    [f'Delta_MFCC_mean_{i}'  for i in mfcc_idx] +
    [f'Delta_MFCC_std_{i}'   for i in mfcc_idx] +
    [f'Delta2_MFCC_mean_{i}' for i in mfcc_idx] +
    [f'Delta2_MFCC_std_{i}'  for i in mfcc_idx] +
    [f'Chroma_mean_{i}'      for i in range(12)] +
    [f'Chroma_std_{i}'       for i in range(12)] +
    ['ZCR_mean', 'ZCR_std'] +
    ['RMS_mean', 'RMS_std']
)

importances = xgb_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# ── Plot 1: Top 20 individual features ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
top20 = importance_df.head(20)
colors = plt.cm.YlOrRd(np.linspace(0.4, 0.9, len(top20)))[::-1]
bars = ax.barh(top20['Feature'][::-1], top20['Importance'][::-1],
               color=colors[::-1], edgecolor='white')
ax.set_title('XGBoost — Top 20 Most Important Features', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, top20['Importance'][::-1]):
    ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

# ── Plot 2: Importance by feature group ──────────────────────────────────────
group_importance = {
    'MFCC mean':         importance_df[importance_df['Feature'].str.startswith('MFCC_mean')]['Importance'].sum(),
    'MFCC std':          importance_df[importance_df['Feature'].str.startswith('MFCC_std')]['Importance'].sum(),
    'Delta MFCC mean':   importance_df[importance_df['Feature'].str.startswith('Delta_MFCC_mean')]['Importance'].sum(),
    'Delta MFCC std':    importance_df[importance_df['Feature'].str.startswith('Delta_MFCC_std')]['Importance'].sum(),
    'Delta2 MFCC mean':  importance_df[importance_df['Feature'].str.startswith('Delta2_MFCC_mean')]['Importance'].sum(),
    'Delta2 MFCC std':   importance_df[importance_df['Feature'].str.startswith('Delta2_MFCC_std')]['Importance'].sum(),
    'Chroma mean':       importance_df[importance_df['Feature'].str.startswith('Chroma_mean')]['Importance'].sum(),
    'Chroma std':        importance_df[importance_df['Feature'].str.startswith('Chroma_std')]['Importance'].sum(),
    'ZCR':               importance_df[importance_df['Feature'].str.startswith('ZCR')]['Importance'].sum(),
    'RMS':               importance_df[importance_df['Feature'].str.startswith('RMS')]['Importance'].sum(),
}

group_df = pd.DataFrame({
    'Group':      list(group_importance.keys()),
    'Importance': list(group_importance.values())
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(group_df)))[::-1]
bars = ax.bar(group_df['Group'], group_df['Importance'],
              color=colors, edgecolor='white')
ax.set_title('XGBoost — Total Importance by Feature Group', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature Group')
ax.set_ylabel('Summed Importance')
plt.xticks(rotation=35, ha='right')
for bar, val in zip(bars, group_df['Importance']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

# ── Plot 3: Full importance heatmap across all MFCC coefficients ─────────────
mfcc_groups = {
    'MFCC mean':        [f'MFCC_mean_{i}'        for i in mfcc_idx],
    'MFCC std':         [f'MFCC_std_{i}'          for i in mfcc_idx],
    'Delta MFCC mean':  [f'Delta_MFCC_mean_{i}'   for i in mfcc_idx],
    'Delta MFCC std':   [f'Delta_MFCC_std_{i}'    for i in mfcc_idx],
    'Delta2 MFCC mean': [f'Delta2_MFCC_mean_{i}'  for i in mfcc_idx],
    'Delta2 MFCC std':  [f'Delta2_MFCC_std_{i}'   for i in mfcc_idx],
}

heatmap_data = pd.DataFrame({
    group: importance_df.set_index('Feature').loc[feats]['Importance'].values
    for group, feats in mfcc_groups.items()
}, index=[f'Coeff {i}' for i in mfcc_idx]).T

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.3, linecolor='white',
            annot=False, ax=ax)
ax.set_title('XGBoost — MFCC Feature Importance Heatmap\n(rows=feature group, cols=coefficient index)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('MFCC Coefficient Index')
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

# ── Summary table: top 10 ─────────────────────────────────────────────────────
print('Top 10 Most Important Features:')
print(importance_df.head(10).to_string(index=False))

## Model 2 — CNN on Mel Spectrograms

In [21]:
def build_cnn(input_shape, n_classes):
    inp = keras.Input(shape=input_shape)

    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    model = keras.Model(inp, out)

    # ── CNOptimizer config ──────────────────────────────────────────
    CNN_OPTIMIZER = 'adamax'
    CNN_LR        = 5e-3     # Learning rate
    
    OPTIMIZERS = {
        'adam':    keras.optimizers.Adam,
        'sgd':     keras.optimizers.SGD,
        'rmsprop': keras.optimizers.RMSprop,
        'adamw':   keras.optimizers.AdamW,
        'adamax':  keras.optimizers.Adamax
    }
    
    optimizer = OPTIMIZERS[CNN_OPTIMIZER](learning_rate=CNN_LR)

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
        
    return model

cnn_input_shape = X_mel_train.shape[1:]   # (N_MELS, MEL_TIME_DIM, 1)
cnn_model = build_cnn(cnn_input_shape, n_classes)
cnn_model.summary()

I0000 00:00:1780104534.582758  665329 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4139 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 65, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 65, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 65, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 8, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 8, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 8, 256)     │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 8, 256)     │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         2,313 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 457,865 (1.75 MB)

 Trainable params: 456,905 (1.74 MB)

 Non-trainable params: 960 (3.75 KB)

In [22]:
cnn_callbacks = [
    callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
]

history_cnn = cnn_model.fit(
    X_mel_train, y_train,
    validation_data=(X_mel_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn_callbacks,
    verbose=1
)

y_pred_cnn = cnn_model.predict(X_mel_test).argmax(axis=1)
acc_cnn    = accuracy_score(y_test, y_pred_cnn)
f1_cnn     = f1_score(y_test, y_pred_cnn, average='weighted')

print(f'\nCNN  →  Accuracy: {acc_cnn:.4f}  |  Weighted F1: {f1_cnn:.4f}')
print()
print(classification_report(y_test, y_pred_cnn, target_names=le.classes_))

Epoch 1/50


I0000 00:00:1780104538.429252  666658 service.cc:153] XLA service 0x78d628040460 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780104538.429275  666658 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.5.0; Toolkit: 12.5.0; DNN: 9.7.1)
I0000 00:00:1780104538.491532  666658 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780104538.724995  666658 cuda_dnn.cc:461] Loaded cuDNN version 90701
I0000 00:00:1780104538.827690  666658 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3937__.61
I0000 00:00:1780104540.970006  666658 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set in

16/43 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1420 - loss: 2.3139

I0000 00:00:1780104548.240993  666658 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


43/43 ━━━━━━━━━━━━━━━━━━━━ 15s 79ms/step - accuracy: 0.3161 - loss: 1.8983 - val_accuracy: 0.0847 - val_loss: 4.1005 - learning_rate: 0.0050
Epoch 2/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5203 - loss: 1.3541 - val_accuracy: 0.1254 - val_loss: 4.8501 - learning_rate: 0.0050
Epoch 3/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6330 - loss: 1.1010 - val_accuracy: 0.0915 - val_loss: 5.1664 - learning_rate: 0.0050
Epoch 4/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6679 - loss: 0.9220 - val_accuracy: 0.0915 - val_loss: 5.7640 - learning_rate: 0.0050
Epoch 5/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7297 - loss: 0.8285 - val_accuracy: 0.0915 - val_loss: 5.1120 - learning_rate: 0.0050
Epoch 6/50
41/43 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7644 - loss: 0.6924
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0024999999441206455.
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7725 - loss: 0.6766 - val_accuracy: 0.09

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cd.plot_training_history(history_cnn, 'CNN (Mel Spectrogram)', axes[0], axes[1])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
cd.plot_confusion_matrix(y_test, y_pred_cnn, le.classes_,
                      'Confusion Matrix — CNN (Mel Spectrogram)', ax)
plt.tight_layout()
plt.show()

## Model 3 — CNN-LSTM Hybrid

In [24]:
def build_cnn_lstm(input_shape, n_classes):
    """
    CNN extracts local frequency features from each time frame;
    LSTM models temporal evolution across frames.
    Input shape: (n_mels, time_steps, 1)
    """
    inp = keras.Input(shape=input_shape)   # (N_MELS, T, 1)

    # CNN block — extract frequency features
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1))(x)       # pool only in frequency dim
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1))(x)
    x = layers.Dropout(0.25)(x)

    # Reshape for LSTM: merge freq and channel dims → (time, features)
    freq_bins, time_steps, channels = x.shape[1], x.shape[2], x.shape[3]
    x = layers.Reshape((time_steps, freq_bins * channels))(x)

    # LSTM block — model temporal patterns
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.3)(x)

    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_lstm_model = build_cnn_lstm(cnn_input_shape, n_classes)
cnn_lstm_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 65, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 65, 64)    │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128, 65, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 64, 65, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64, 65, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 65, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64, 65, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 32, 65, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32, 65, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 65, 4096)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 65, 256)        │     4,326,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 65, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,583,689 (17.49 MB)

 Trainable params: 4,583,305 (17.48 MB)

 Non-trainable params: 384 (1.50 KB)

In [25]:
cnn_lstm_callbacks = [
    callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
]

history_cnn_lstm = cnn_lstm_model.fit(
    X_mel_train, y_train,
    validation_data=(X_mel_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn_lstm_callbacks,
    verbose=1
)

y_pred_cnn_lstm = cnn_lstm_model.predict(X_mel_test).argmax(axis=1)
acc_cnn_lstm    = accuracy_score(y_test, y_pred_cnn_lstm)
f1_cnn_lstm     = f1_score(y_test, y_pred_cnn_lstm, average='weighted')

print(f'\nCNN-LSTM  →  Accuracy: {acc_cnn_lstm:.4f}  |  Weighted F1: {f1_cnn_lstm:.4f}')
print()
print(classification_report(y_test, y_pred_cnn_lstm, target_names=le.classes_))

Epoch 1/50


E0000 00:00:1780104573.052977  665329 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/dropout_4_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


43/43 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - accuracy: 0.2224 - loss: 2.0573 - val_accuracy: 0.1763 - val_loss: 2.1811 - learning_rate: 0.0010
Epoch 2/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 69ms/step - accuracy: 0.3110 - loss: 1.8327 - val_accuracy: 0.1186 - val_loss: 2.2930 - learning_rate: 0.0010
Epoch 3/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.3859 - loss: 1.6517 - val_accuracy: 0.1085 - val_loss: 2.3145 - learning_rate: 0.0010
Epoch 4/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.4310 - loss: 1.5535 - val_accuracy: 0.1119 - val_loss: 2.4001 - learning_rate: 0.0010
Epoch 5/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.5182 - loss: 1.3237 - val_accuracy: 0.1322 - val_loss: 2.8260 - learning_rate: 0.0010
Epoch 6/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.5738 - loss: 1.2537
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.5879 - loss: 1.2036 - val_accuracy: 0.132

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cd.plot_training_history(history_cnn_lstm, 'CNN-LSTM Hybrid', axes[0], axes[1])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
cd.plot_confusion_matrix(y_test, y_pred_cnn_lstm, le.classes_,
                      'Confusion Matrix — CNN-LSTM Hybrid', ax)
plt.tight_layout()
plt.show()

## Model 4 — Transfer Learning with YAMNet

In [27]:
# Load YAMNet from TensorFlow Hub (requires internet on first run)
YAMNET_URL = 'https://tfhub.dev/google/yamnet/1'
print('Loading YAMNet from TF Hub...')
yamnet_model = hub.load(YAMNET_URL)
print('YAMNet loaded.')

Loading YAMNet from TF Hub...
YAMNet loaded.


In [28]:
def extract_yamnet_embeddings(paths, sr=SR, batch_size=16):
    """
    Pass each waveform through YAMNet and return the mean embedding
    over all frames (shape: N x 1024).
    """
    embeddings = []
    for path in tqdm(paths, desc='YAMNet embeddings'):
        y = extract_yamnet_waveform(path)
        # YAMNet returns (scores, embeddings, log_mel)
        _, emb, _ = yamnet_model(y)
        embeddings.append(emb.numpy().mean(axis=0))   # mean-pool over time
    return np.array(embeddings)

all_paths = df['path'].tolist()
print('Extracting YAMNet embeddings for all files...')
X_yamnet = extract_yamnet_embeddings(all_paths)
print(f'YAMNet embedding matrix: {X_yamnet.shape}')

Extracting YAMNet embeddings for all files...


YAMNet embeddings: 100%|██████████████████████████████████████████████████████| 1967/1967 [00:06<00:00, 300.23it/s]

YAMNet embedding matrix: (1967, 1024)


In [29]:
# Reuse the same splits
X_yamnet_train = X_yamnet[idx_train]
X_yamnet_val   = X_yamnet[idx_val]
X_yamnet_test  = X_yamnet[idx_test]

def build_yamnet_head(embedding_dim, n_classes):
    """Lightweight MLP classifier on top of frozen YAMNet embeddings."""
    model = keras.Sequential([
        keras.Input(shape=(embedding_dim,)),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adamax(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

yamnet_head = build_yamnet_head(X_yamnet.shape[1], n_classes)
yamnet_head.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 9)              │         2,313 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 661,513 (2.52 MB)

 Trainable params: 659,977 (2.52 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [30]:
yamnet_callbacks = [
    callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(patience=8, factor=0.5, verbose=1),
]

history_yamnet = yamnet_head.fit(
    X_yamnet_train, y_train,
    validation_data=(X_yamnet_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=yamnet_callbacks,
    verbose=1
)

y_pred_yamnet = yamnet_head.predict(X_yamnet_test).argmax(axis=1)
acc_yamnet    = accuracy_score(y_test, y_pred_yamnet)
f1_yamnet     = f1_score(y_test, y_pred_yamnet, average='weighted')

print(f'\nYAMNet  →  Accuracy: {acc_yamnet:.4f}  |  Weighted F1: {f1_yamnet:.4f}')
print()
print(classification_report(y_test, y_pred_yamnet, target_names=le.classes_))

Epoch 1/50


I0000 00:00:1780104626.175743  666658 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_44262__.23


43/43 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.5094 - loss: 1.5464 - val_accuracy: 0.6136 - val_loss: 1.6863 - learning_rate: 0.0010
Epoch 2/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6897 - loss: 0.9748 - val_accuracy: 0.6644 - val_loss: 1.5173 - learning_rate: 0.0010
Epoch 3/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7297 - loss: 0.8223 - val_accuracy: 0.6983 - val_loss: 1.3644 - learning_rate: 0.0010
Epoch 4/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7645 - loss: 0.6970 - val_accuracy: 0.7017 - val_loss: 1.2284 - learning_rate: 0.0010
Epoch 5/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7951 - loss: 0.6150 - val_accuracy: 0.7186 - val_loss: 1.0856 - learning_rate: 0.0010
Epoch 6/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8089 - loss: 0.5757 - val_accuracy: 0.7288 - val_loss: 0.9675 - learning_rate: 0.0010
Epoch 7/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8350 - loss: 0.5027 - val_accuracy: 0.7390 - v

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cd.plot_training_history(history_yamnet, 'YAMNet Transfer Learning', axes[0], axes[1])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
cd.plot_confusion_matrix(y_test, y_pred_yamnet, le.classes_,
                      'Confusion Matrix — YAMNet Transfer Learning', ax)
plt.tight_layout()
plt.show()

## Model 5 - Transfer Learning with PANNs

In [32]:
# !pip install panns-inference
from panns_inference import AudioTagging

print('Loading PANNs CNN14 model...')
panns_model = AudioTagging(checkpoint_path=None, device='cuda')  # use 'cpu' if no GPU available
print('PANNs CNN14 loaded.')

Loading PANNs CNN14 model...
Checkpoint path: /home/jose/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1
PANNs CNN14 loaded.


In [33]:
def extract_panns_embeddings(paths, sr=SR):
    """
    Pass each waveform through PANNs CNN14 and return the
    clipwise embedding (2048-dim) for each file.
    """
    embeddings = []
    for path in tqdm(paths, desc='PANNs embeddings'):
        y, _ = librosa.load(path, sr=32000, mono=True)   # PANNs expects 32kHz
        y = y[np.newaxis, :]                              # add batch dim → (1, samples)
        _, emb = panns_model.inference(y)                 # emb shape: (1, 2048)
        embeddings.append(emb[0])
    return np.array(embeddings)

all_paths = df['path'].tolist()
print('Extracting PANNs embeddings for all files...')
X_panns = extract_panns_embeddings(all_paths)
print(f'PANNs embedding matrix: {X_panns.shape}')   # (N, 2048)

Extracting PANNs embeddings for all files...


PANNs embeddings: 100%|████████████████████████████████████████████████████████| 1967/1967 [00:25<00:00, 77.35it/s]

PANNs embedding matrix: (1967, 2048)


In [34]:
X_panns_train = X_panns[idx_train]
X_panns_val   = X_panns[idx_val]
X_panns_test  = X_panns[idx_test]

print(f'Train: {X_panns_train.shape}, Val: {X_panns_val.shape}, Test: {X_panns_test.shape}')

Train: (1376, 2048), Val: (295, 2048), Test: (296, 2048)


In [41]:
from tensorflow.keras import regularizers

L2_LAMBDA = 1e-4   # same as Model 5

def build_panns_head(embedding_dim, n_classes):
    model = keras.Sequential([
        keras.Input(shape=(embedding_dim,)),
        layers.Dense(1024, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.6),
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adamax(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

panns_head = build_panns_head(X_panns.shape[1], n_classes)
panns_head.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_11 (Dense)                │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 9)              │         2,313 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,763,785 (10.54 MB)

 Trainable params: 2,760,201 (10.53 MB)

 Non-trainable params: 3,584 (14.00 KB)

In [42]:
panns_callbacks = [
    callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(patience=10, factor=0.5, verbose=1),
]

history_panns = panns_head.fit(
    X_panns_train, y_train,
    validation_data=(X_panns_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=panns_callbacks,
    verbose=1
)

y_pred_panns = panns_head.predict(X_panns_test).argmax(axis=1)
acc_panns    = accuracy_score(y_test, y_pred_panns)
f1_panns     = f1_score(y_test, y_pred_panns, average='weighted')

print(f'\nPANNs CNN14  →  Accuracy: {acc_panns:.4f}  |  Weighted F1: {f1_panns:.4f}')
print()
print(classification_report(y_test, y_pred_panns, target_names=le.classes_))

Epoch 1/50


I0000 00:00:1780104829.715887  666654 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_73284__.33


43/43 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.4426 - loss: 2.0297 - val_accuracy: 0.5627 - val_loss: 1.7780 - learning_rate: 0.0010
Epoch 2/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6257 - loss: 1.3485 - val_accuracy: 0.6610 - val_loss: 1.5450 - learning_rate: 0.0010
Epoch 3/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6548 - loss: 1.2178 - val_accuracy: 0.7085 - val_loss: 1.3613 - learning_rate: 0.0010
Epoch 4/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6919 - loss: 1.1179 - val_accuracy: 0.7288 - val_loss: 1.1818 - learning_rate: 0.0010
Epoch 5/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7078 - loss: 1.0418 - val_accuracy: 0.7458 - val_loss: 1.0644 - learning_rate: 0.0010
Epoch 6/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7275 - loss: 0.9971 - val_accuracy: 0.7695 - val_loss: 0.9417 - learning_rate: 0.0010
Epoch 7/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7594 - loss: 0.9264 - val_accuracy: 0.8000 - v

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cd.plot_training_history(history_panns, 'PANNs CNN14 Transfer Learning', axes[0], axes[1])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
cd.plot_confusion_matrix(y_test, y_pred_panns, le.classes_,
                      'Confusion Matrix — PANNs CNN14', ax)
plt.tight_layout()
plt.show()

## Model 6 - Hierarchical PANNs Mixture-of-Experts
### Setup: State/Subclass Mappings

In [76]:
# Model 6: Hierarchical PANNs Mixture of Experts
# Gating model classifies into 3 states; each expert classifies subcategories.

# State-to-subclass mappings (must match your dataset folder names)
STATE_SUBCLASSES = {
    'braking_state':  ['normal_brakes',         'worn_out_brakes'],
    'idle_state':     ['low_oil', 'normal_engine_idle', 'power_steering', 'serpentine_belt'],
    'startup_state':  ['bad_ignition',           'dead_battery',  'normal_engine_startup'],
}
STATES = ['braking_state', 'idle_state', 'startup_state']

# Label encoders — one for the gating model, one per expert
from sklearn.preprocessing import LabelEncoder

le_state = LabelEncoder()
le_state.fit(STATES)

le_experts = {}
for state, subclasses in STATE_SUBCLASSES.items():
    le_exp = LabelEncoder()
    le_exp.fit(subclasses)
    le_experts[state] = le_exp

print('State label encoder classes:', list(le_state.classes_))
for state, le_exp in le_experts.items():
    print(f'  Expert [{state}] classes: {list(le_exp.classes_)}')

State label encoder classes: [np.str_('braking_state'), np.str_('idle_state'), np.str_('startup_state')]
  Expert [braking_state] classes: [np.str_('normal_brakes'), np.str_('worn_out_brakes')]
  Expert [idle_state] classes: [np.str_('low_oil'), np.str_('normal_engine_idle'), np.str_('power_steering'), np.str_('serpentine_belt')]
  Expert [startup_state] classes: [np.str_('bad_ignition'), np.str_('dead_battery'), np.str_('normal_engine_startup')]


### Reuse PANNs Embeddings (already extracted as X_panns)

In [77]:
# X_panns was extracted in Model 5: shape (N, 2048), using the same df row order.
# We just need state labels and per-state subsets.

# State labels for the gating model
y_state_raw = df['state'].values                         # e.g. 'braking_state'
y_state_enc = le_state.transform(y_state_raw)

# Gating model splits (reuse idx_train / idx_val / idx_test from the global split)
X_gate_train = X_panns[idx_train];   y_gate_train = y_state_enc[idx_train]
X_gate_val   = X_panns[idx_val];     y_gate_val   = y_state_enc[idx_val]
X_gate_test  = X_panns[idx_test];    y_gate_test  = y_state_enc[idx_test]

print(f'Gating — train: {X_gate_train.shape}, val: {X_gate_val.shape}, test: {X_gate_test.shape}')
print(f'State distribution in test set: {dict(zip(le_state.classes_, np.bincount(y_gate_test)))}')

Gating — train: (1376, 2048), val: (295, 2048), test: (296, 2048)
State distribution in test set: {np.str_('braking_state'): np.int64(69), np.str_('idle_state'): np.int64(146), np.str_('startup_state'): np.int64(81)}


### Train the Gating Model

In [80]:
from tensorflow.keras import regularizers

L2_LAMBDA = 1e-4   # same as Model 5

def build_gating_model(embedding_dim=2048, n_states=3):
    """
    3-way classifier: braking_state / idle_state / startup_state.
    Slightly shallower than the full 9-class head — the task is easier.
    """
    model = keras.Sequential([
        keras.Input(shape=(embedding_dim,)),
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(n_states, activation='softmax'),
    ], name='gating_model')
    model.compile(
        optimizer=keras.optimizers.Adamax(5e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

gate_model = build_gating_model()
gate_model.summary()

Model: "gating_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_31 (Dense)                │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_29 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_30 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,217,283 (4.64 MB)

 Trainable params: 1,215,491 (4.64 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [81]:
gate_callbacks = [
    callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
]

history_gate = gate_model.fit(
    X_gate_train, y_gate_train,
    validation_data=(X_gate_val, y_gate_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=gate_callbacks,
    verbose=1
)

y_pred_gate_test = gate_model.predict(X_gate_test).argmax(axis=1)
acc_gate = accuracy_score(y_gate_test, y_pred_gate_test)
f1_gate  = f1_score(y_gate_test, y_pred_gate_test, average='weighted')

print(f'\nGating Model  →  Accuracy: {acc_gate:.4f}  |  Weighted F1: {f1_gate:.4f}')
print()
print(classification_report(y_gate_test, y_pred_gate_test, target_names=le_state.classes_))

Epoch 1/50


I0000 00:00:1780106938.375827  666654 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_148862__.33


43/43 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.7551 - loss: 0.7950 - val_accuracy: 0.7220 - val_loss: 0.7259 - learning_rate: 0.0050
Epoch 2/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8445 - loss: 0.4933 - val_accuracy: 0.7661 - val_loss: 0.6270 - learning_rate: 0.0050
Epoch 3/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8757 - loss: 0.4349 - val_accuracy: 0.8102 - val_loss: 0.5513 - learning_rate: 0.0050
Epoch 4/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8852 - loss: 0.3873 - val_accuracy: 0.8407 - val_loss: 0.4760 - learning_rate: 0.0050
Epoch 5/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8990 - loss: 0.3633 - val_accuracy: 0.8644 - val_loss: 0.4464 - learning_rate: 0.0050
Epoch 6/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9041 - loss: 0.3362 - val_accuracy: 0.8712 - val_loss: 0.4131 - learning_rate: 0.0050
Epoch 7/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9142 - loss: 0.3147 - val_accuracy: 0.8814 - v

### Train One Expert per State

In [82]:
def build_expert_model(embedding_dim=2048, n_subclasses=2, name='expert'):
    """
    Subclass classifier for a single state.
    The head depth scales with the number of subclasses.
    """
    model = keras.Sequential([
        keras.Input(shape=(embedding_dim,)),
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(n_subclasses, activation='softmax'),
    ], name=name)
    model.compile(
        optimizer=keras.optimizers.Adamax(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


expert_models   = {}   # state → trained Keras model
history_experts = {}   # state → history object

# Ground-truth fine-grained label for every sample (needed to train each expert)
y_fine_raw = df['label'].values   # 9 subclasses

for state in STATES:
    print(f'\n{"="*60}')
    print(f'  Training expert: {state}')
    print(f'{"="*60}')

    # Indices (within the full dataset) that belong to this state
    state_mask = (df['state'].values == state)

    # Build per-split masks
    # We work in the index space of the global split arrays.
    train_mask = state_mask[idx_train]
    val_mask   = state_mask[idx_val]
    test_mask  = state_mask[idx_test]

    X_exp_train = X_panns_train[train_mask]
    X_exp_val   = X_panns_val[val_mask]

    # Encode subclass labels for this state
    le_exp = le_experts[state]
    y_exp_train = le_exp.transform(y_fine_raw[idx_train][train_mask])
    y_exp_val   = le_exp.transform(y_fine_raw[idx_val][val_mask])

    print(f'  Samples — train: {len(X_exp_train)}, val: {len(X_exp_val)}')
    print(f'  Subclasses: {list(le_exp.classes_)}')

    n_sub = len(le_exp.classes_)
    model = build_expert_model(
        embedding_dim=X_panns.shape[1],
        n_subclasses=n_sub,
        name=f'expert_{state}'
    )

    exp_callbacks = [
        callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
    ]

    hist = model.fit(
        X_exp_train, y_exp_train,
        validation_data=(X_exp_val, y_exp_val),
        epochs=EPOCHS,
        batch_size=min(BATCH_SIZE, max(8, len(X_exp_train) // 4)),
        callbacks=exp_callbacks,
        verbose=1
    )

    expert_models[state]   = model
    history_experts[state] = hist

print('\nAll expert models trained.')


  Training expert: braking_state
  Samples — train: 321, val: 69
  Subclasses: [np.str_('normal_brakes'), np.str_('worn_out_brakes')]
Epoch 1/50


I0000 00:00:1780106981.659851  666647 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_159808__.39


11/11 ━━━━━━━━━━━━━━━━━━━━ 8s 358ms/step - accuracy: 0.7944 - loss: 0.6628 - val_accuracy: 0.9275 - val_loss: 0.6397 - learning_rate: 0.0010
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8816 - loss: 0.3728 - val_accuracy: 0.9130 - val_loss: 0.5682 - learning_rate: 0.0010
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9190 - loss: 0.3114 - val_accuracy: 0.9275 - val_loss: 0.5300 - learning_rate: 0.0010
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9097 - loss: 0.3254 - val_accuracy: 0.9130 - val_loss: 0.5131 - learning_rate: 0.0010
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9221 - loss: 0.2790 - val_accuracy: 0.9130 - val_loss: 0.4893 - learning_rate: 0.0010
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9034 - loss: 0.3062 - val_accuracy: 0.8986 - val_loss: 0.4749 - learning_rate: 0.0010
Epoch 7/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9439 - loss: 0.2160 - val_accuracy: 0.9130 - 

I0000 00:00:1780106993.657689  666657 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_167716__.39


 1/22 ━━━━━━━━━━━━━━━━━━━━ 1:31 4s/step - accuracy: 0.2500 - loss: 2.1184

...

22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 209ms/step - accuracy: 0.5746 - loss: 1.3175 - val_accuracy: 0.6414 - val_loss: 1.1590 - learning_rate: 0.0010
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7267 - loss: 0.7966 - val_accuracy: 0.7793 - val_loss: 1.0616 - learning_rate: 0.0010
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7755 - loss: 0.7159 - val_accuracy: 0.7724 - val_loss: 1.0166 - learning_rate: 0.0010
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7888 - loss: 0.7226 - val_accuracy: 0.8000 - val_loss: 0.9345 - learning_rate: 0.0010
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8021 - loss: 0.6481 - val_accuracy: 0.8345 - val_loss: 0.8555 - learning_rate: 0.0010
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8006 - loss: 0.6400 - val_accuracy: 0.8345 - val_loss: 0.7973 - learning_rate: 0.0010
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8346 - loss: 0.5534 - val_accuracy: 0.8621 - 

I0000 00:00:1780107007.233117  666652 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_177108__.39


 1/12 ━━━━━━━━━━━━━━━━━━━━ 42s 4s/step - accuracy: 0.4375 - loss: 1.5721

I0000 00:00:1780107009.584283  666658 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_177108__.39


12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 432ms/step - accuracy: 0.5899 - loss: 1.1637 - val_accuracy: 0.7778 - val_loss: 0.9607 - learning_rate: 0.0010
Epoch 2/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7963 - loss: 0.6804 - val_accuracy: 0.7901 - val_loss: 0.8850 - learning_rate: 0.0010
Epoch 3/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8307 - loss: 0.5235 - val_accuracy: 0.8025 - val_loss: 0.8176 - learning_rate: 0.0010
Epoch 4/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8783 - loss: 0.4269 - val_accuracy: 0.8148 - val_loss: 0.7636 - learning_rate: 0.0010
Epoch 5/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9021 - loss: 0.4065 - val_accuracy: 0.8272 - val_loss: 0.7208 - learning_rate: 0.0010
Epoch 6/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8968 - loss: 0.4022 - val_accuracy: 0.8272 - val_loss: 0.6784 - learning_rate: 0.0010
Epoch 7/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9101 - loss: 0.3419 - val_accuracy: 0.8272 - 

### Hierarchical Inference & Evaluation

In [83]:
def hierarchical_predict(X_emb, gate_model, expert_models, le_state, le_experts):
    """
    Two-stage prediction:
      1. Gating model → predicted state (hard routing).
      2. Corresponding expert → predicted subclass.

    Returns
    -------
    y_pred_state    : np.ndarray of shape (N,)  — predicted state indices (le_state encoding)
    y_pred_fine_str : list of str               — predicted fine-grained label strings
    """
    gate_probs   = gate_model.predict(X_emb, verbose=0)   # (N, 3)
    pred_states  = gate_probs.argmax(axis=1)               # hard routing
    state_names  = le_state.inverse_transform(pred_states)

    pred_fine = []
    for i, state in enumerate(state_names):
        emb       = X_emb[i:i+1]                              # (1, 2048)
        sub_probs = expert_models[state].predict(emb, verbose=0)  # (1, n_sub)
        sub_idx   = sub_probs.argmax(axis=1)[0]
        fine_label = le_experts[state].inverse_transform([sub_idx])[0]
        pred_fine.append(fine_label)

    return pred_states, pred_fine


# Evaluate on test set
print('Running hierarchical inference on test set...')

X_test_emb = X_panns_test   # (N_test, 2048)

y_pred_state_hier, y_pred_fine_hier = hierarchical_predict(
    X_test_emb, gate_model, expert_models, le_state, le_experts
)

# Fine-grained ground truth (string labels) for the test set
y_test_fine_str  = y_fine_raw[idx_test]
y_test_state_str = df['state'].values[idx_test]

# Convert predictions to label-encoded ints for sklearn metrics
# (reuse the global le from Model 1–5 which encodes all 9 subclasses)
y_pred_fine_enc = le.transform(y_pred_fine_hier)
y_test_fine_enc = le.transform(y_test_fine_str)

acc_hier = accuracy_score(y_test_fine_enc, y_pred_fine_enc)
f1_hier  = f1_score(y_test_fine_enc, y_pred_fine_enc, average='weighted')

print(f'\nHierarchical PANNs (MoE)  →  Accuracy: {acc_hier:.4f}  |  Weighted F1: {f1_hier:.4f}')
print()
print(classification_report(y_test_fine_enc, y_pred_fine_enc, target_names=le.classes_))

Running hierarchical inference on test set...

Hierarchical PANNs (MoE)  →  Accuracy: 0.8378  |  Weighted F1: 0.8388

                       precision    recall  f1-score   support

         bad_ignition       0.92      0.86      0.89        28
         dead_battery       0.96      0.88      0.92        26
              low_oil       0.82      0.72      0.77        32
        normal_brakes       0.78      0.83      0.81        35
   normal_engine_idle       0.81      0.95      0.87        40
normal_engine_startup       0.92      0.89      0.91        27
       power_steering       0.69      0.79      0.74        39
      serpentine_belt       0.93      0.80      0.86        35
      worn_out_brakes       0.85      0.82      0.84        34

             accuracy                           0.84       296
            macro avg       0.85      0.84      0.84       296
         weighted avg       0.85      0.84      0.84       296



### Visualisations

In [ ]:
# Training curves: gating model
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cd.plot_training_history(history_gate, 'Gating Model (3 States)', axes[0], axes[1])
plt.tight_layout()
plt.show()

# Training curves: all three experts
fig, axes = plt.subplots(3, 2, figsize=(13, 10))
fig.suptitle('Expert Model Training Curves', fontsize=14, fontweight='bold')
for row_ax, state in zip(axes, STATES):
    cd.plot_training_history(history_experts[state],
                          f'Expert — {state}', row_ax[0], row_ax[1])
plt.tight_layout()
plt.show()

In [ ]:
# Gating confusion matrix
fig, ax = plt.subplots(figsize=(5, 3))
plot_confusion_matrix(
    le_state.transform(y_test_state_str),
    y_pred_state_hier,
    le_state.classes_,
    'Gating Model — State Confusion Matrix',
    ax
)
plt.tight_layout()
plt.show()

# Fine-grained confusion matrix (end-to-end)
fig, ax = plt.subplots(figsize=(8, 5))
plot_confusion_matrix(
    y_test_fine_enc, y_pred_fine_enc,
    le.classes_,
    f'Hierarchical PANNs MoE — End-to-End\nAcc={acc_hier:.4f} | F1={f1_hier:.4f}',
    ax
)
plt.tight_layout()
plt.show()

In [86]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_fine_enc, y_pred_fine_xgb_enc)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
display(cm_df)

,bad_ignition,dead_battery,low_oil,normal_brakes,normal_engine_idle,normal_engine_startup,power_steering,serpentine_belt,worn_out_brakes
bad_ignition,23,0,1,1,0,1,1,0,1
dead_battery,0,25,1,0,0,0,0,0,0
low_oil,1,1,24,1,3,1,1,0,0
normal_brakes,0,0,0,31,1,0,0,0,3
normal_engine_idle,0,0,0,1,38,0,1,0,0
normal_engine_startup,0,0,1,0,1,24,1,0,0
power_steering,0,0,1,0,3,0,33,1,1
serpentine_belt,0,0,3,0,0,1,1,30,0
worn_out_brakes,0,2,1,1,0,1,1,0,28


## Hierarchical XGBoost Mixture-of-Experts
### Train the XGBoost Gating Model

In [60]:
print('Training XGBoost gating model (3 states)...')

# State labels for the gating model (reuses X_mfcc_* splits from Section 4)
y_state_enc_full = le_state.transform(df['state'].values)
y_gate_train_xgb = y_state_enc_full[idx_train]
y_gate_val_xgb   = y_state_enc_full[idx_val]
y_gate_test_xgb  = y_state_enc_full[idx_test]

gate_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1,
    early_stopping_rounds=20,
)
gate_xgb.fit(
    X_mfcc_train_s, y_gate_train_xgb,
    eval_set=[(X_mfcc_val_s, y_gate_val_xgb)],
    verbose=False
)

y_pred_gate_xgb = gate_xgb.predict(X_mfcc_test_s)
acc_gate_xgb = accuracy_score(y_gate_test_xgb, y_pred_gate_xgb)
f1_gate_xgb  = f1_score(y_gate_test_xgb, y_pred_gate_xgb, average='weighted')

print(f'XGBoost Gating  →  Accuracy: {acc_gate_xgb:.4f}  |  Weighted F1: {f1_gate_xgb:.4f}')
print()
print(classification_report(y_gate_test_xgb, y_pred_gate_xgb, target_names=le_state.classes_))

Training XGBoost gating model (3 states)...
XGBoost Gating  →  Accuracy: 0.9291  |  Weighted F1: 0.9289

               precision    recall  f1-score   support

braking_state       0.93      0.91      0.92        69
   idle_state       0.94      0.95      0.95       146
startup_state       0.91      0.90      0.91        81

     accuracy                           0.93       296
    macro avg       0.93      0.92      0.92       296
 weighted avg       0.93      0.93      0.93       296



### Train One XGBoost Expert per State

In [63]:
expert_xgb_models = {}   # state → trained XGBClassifier

y_fine_full = df['label'].values   # fine-grained string labels for all rows

for state in STATES:
    print(f'\n{"="*60}')
    print(f'  Training XGBoost expert: {state}')
    print(f'{"="*60}')

    state_mask  = (df['state'].values == state)
    train_mask  = state_mask[idx_train]
    val_mask    = state_mask[idx_val]

    X_exp_train = X_mfcc_train_s[train_mask]
    X_exp_val   = X_mfcc_val_s[val_mask]

    le_exp      = le_experts[state]
    y_exp_train = le_exp.transform(y_fine_full[idx_train][train_mask])
    y_exp_val   = le_exp.transform(y_fine_full[idx_val][val_mask])

    n_sub       = len(le_exp.classes_)
    print(f'  Samples — train: {len(X_exp_train)}, val: {len(X_exp_val)}')
    print(f'  Subclasses ({n_sub}): {list(le_exp.classes_)}')

    # Binary experts use logloss; multiclass experts use mlogloss
    eval_metric = 'logloss' if n_sub == 2 else 'mlogloss'
    objective   = 'binary:logistic' if n_sub == 2 else 'multi:softprob'

    expert_xgb = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        eval_metric=eval_metric,
        num_class=n_sub if n_sub > 2 else None,
        use_label_encoder=False,
        random_state=SEED,
        n_jobs=-1,
        early_stopping_rounds=20,
    )
    expert_xgb.fit(
        X_exp_train, y_exp_train,
        eval_set=[(X_exp_val, y_exp_val)],
        verbose=False
    )

    expert_xgb_models[state] = expert_xgb
    print(f'  Done.')

print('\nAll XGBoost expert models trained.')


  Training XGBoost expert: braking_state
  Samples — train: 321, val: 69
  Subclasses (2): [np.str_('normal_brakes'), np.str_('worn_out_brakes')]
  Done.

  Training XGBoost expert: idle_state
  Samples — train: 677, val: 145
  Subclasses (4): [np.str_('low_oil'), np.str_('normal_engine_idle'), np.str_('power_steering'), np.str_('serpentine_belt')]
  Done.

  Training XGBoost expert: startup_state
  Samples — train: 378, val: 81
  Subclasses (3): [np.str_('bad_ignition'), np.str_('dead_battery'), np.str_('normal_engine_startup')]
  Done.

All XGBoost expert models trained.


### Hierarchical Inference & Evaluation

In [64]:
def hierarchical_predict_xgb(X_mfcc_scaled, gate_model, expert_models,
                              le_state, le_experts):
    """
    Two-stage XGBoost prediction:
      1. Gating model → predicted state (hard routing on argmax of predict_proba).
      2. Corresponding expert → predicted subclass.
    """
    gate_probs  = gate_model.predict_proba(X_mfcc_scaled)   # (N, 3)
    pred_states = gate_probs.argmax(axis=1)
    state_names = le_state.inverse_transform(pred_states)

    pred_fine = []
    for i, state in enumerate(state_names):
        sub_probs  = expert_models[state].predict_proba(X_mfcc_scaled[i:i+1])
        sub_idx    = sub_probs.argmax(axis=1)[0]
        fine_label = le_experts[state].inverse_transform([sub_idx])[0]
        pred_fine.append(fine_label)

    return pred_states, pred_fine


print('Running hierarchical XGBoost inference on test set...')

y_pred_state_xgb_hier, y_pred_fine_xgb_hier = hierarchical_predict_xgb(
    X_mfcc_test_s, gate_xgb, expert_xgb_models, le_state, le_experts
)

y_test_fine_str  = y_fine_full[idx_test]
y_pred_fine_xgb_enc = le.transform(y_pred_fine_xgb_hier)
y_test_fine_enc     = le.transform(y_test_fine_str)

acc_xgb_hier = accuracy_score(y_test_fine_enc, y_pred_fine_xgb_enc)
f1_xgb_hier  = f1_score(y_test_fine_enc, y_pred_fine_xgb_enc, average='weighted')

print(f'\nHierarchical XGBoost (MoE)  →  Accuracy: {acc_xgb_hier:.4f}  |  Weighted F1: {f1_xgb_hier:.4f}')
print()
print(classification_report(y_test_fine_enc, y_pred_fine_xgb_enc, target_names=le.classes_))

Running hierarchical XGBoost inference on test set...

Hierarchical XGBoost (MoE)  →  Accuracy: 0.8649  |  Weighted F1: 0.8648

                       precision    recall  f1-score   support

         bad_ignition       0.96      0.82      0.88        28
         dead_battery       0.89      0.96      0.93        26
              low_oil       0.75      0.75      0.75        32
        normal_brakes       0.89      0.89      0.89        35
   normal_engine_idle       0.83      0.95      0.88        40
normal_engine_startup       0.86      0.89      0.87        27
       power_steering       0.85      0.85      0.85        39
      serpentine_belt       0.97      0.86      0.91        35
      worn_out_brakes       0.85      0.82      0.84        34

             accuracy                           0.86       296
            macro avg       0.87      0.86      0.87       296
         weighted avg       0.87      0.86      0.86       296



### Visualisations

In [ ]:
# Gating confusion matrix
fig, ax = plt.subplots(figsize=(5, 3))
plot_confusion_matrix(
    y_gate_test_xgb,
    y_pred_gate_xgb,
    le_state.classes_,
    'XGBoost Gating Model — State Confusion Matrix',
    ax
)
plt.tight_layout()
plt.show()

# End-to-end fine-grained confusion matrix
fig, ax = plt.subplots(figsize=(8, 5))
plot_confusion_matrix(
    y_test_fine_enc,
    y_pred_fine_xgb_enc,
    le.classes_,
    f'Hierarchical XGBoost MoE — End-to-End\nAcc={acc_xgb_hier:.4f} | F1={f1_xgb_hier:.4f}',
    ax
)
plt.tight_layout()
plt.show()

In [69]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_fine_enc, y_pred_fine_xgb_enc)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
display(cm_df)

,bad_ignition,dead_battery,low_oil,normal_brakes,normal_engine_idle,normal_engine_startup,power_steering,serpentine_belt,worn_out_brakes
bad_ignition,23,0,1,1,0,1,1,0,1
dead_battery,0,25,1,0,0,0,0,0,0
low_oil,1,1,24,1,3,1,1,0,0
normal_brakes,0,0,0,31,1,0,0,0,3
normal_engine_idle,0,0,0,1,38,0,1,0,0
normal_engine_startup,0,0,1,0,1,24,1,0,0
power_steering,0,0,1,0,3,0,33,1,1
serpentine_belt,0,0,3,0,0,1,1,30,0
worn_out_brakes,0,2,1,1,0,1,1,0,28


## Ensemble: Majority-Vote (Top-3 Models)

The three highest-performing models — **XGBoost (MFCCs)**, **Mixture-of-Experts (PANNs)**, and **Mixture-of-Experts (XGB)** — vote on each test sample.
A hard majority vote (2-of-3) determines the final prediction; ties are broken by choosing the class with the highest summed probability / confidence across all three voters.

In [90]:
# ─────────────────────────────────────────────────────────────────────
# Ensemble: Hard Majority Vote — Top-3 Models
# Models: XGBoost (MFCCs)  |  MoE PANNs  |  MoE XGBoost
#
# Each model casts a hard vote (integer class index in `le` encoding).
# The class that appears most often wins; ties are broken by the
# sum of calibrated probabilities across voters.
# ─────────────────────────────────────────────────────────────────────

from scipy import stats

# ── Collect hard predictions from the three voters ────────────────────
# (all already in `le`-encoded integer space)
votes_xgb      = y_pred_xgb            # shape (N_test,)  — Model 1: XGBoost (MFCCs)
votes_moe_panns = y_pred_fine_enc       # shape (N_test,)  — Model 6: MoE PANNs
votes_moe_xgb   = y_pred_fine_xgb_enc  # shape (N_test,)  — Model 7: MoE XGBoost

# Stack votes: shape (N_test, 3)
all_votes = np.stack([votes_xgb, votes_moe_panns, votes_moe_xgb], axis=1)

# ── Soft probabilities for tie-breaking ──────────────────────────────
# XGBoost (MFCCs): already outputs probabilities over 9 classes
proba_xgb = xgb_model.predict_proba(X_mfcc_test_s)         # (N, 9)

# MoE PANNs: reconstruct a 9-class probability vector from the
# two-stage gating + expert outputs.
# For each test sample we use the predicted state and the expert's
# sub-class probabilities to fill the relevant slots.
def moe_panns_proba(X_emb, gate_model, expert_models, le_state, le_experts, le_global):
    """Return (N, n_global_classes) soft probability matrix for PANNs MoE."""
    n_global = len(le_global.classes_)
    gate_probs  = gate_model.predict(X_emb, verbose=0)   # (N, 3)
    pred_states = gate_probs.argmax(axis=1)
    state_names = le_state.inverse_transform(pred_states)

    out = np.zeros((len(X_emb), n_global), dtype=np.float32)
    for i, state in enumerate(state_names):
        emb       = X_emb[i:i+1]
        sub_probs = expert_models[state].predict(emb, verbose=0)[0]   # (n_sub,)
        for local_idx, subclass in enumerate(le_experts[state].classes_):
            global_idx = le_global.transform([subclass])[0]
            out[i, global_idx] = sub_probs[local_idx]
    return out

def moe_xgb_proba(X_mfcc_scaled, gate_model, expert_models, le_state, le_experts, le_global):
    """Return (N, n_global_classes) soft probability matrix for XGBoost MoE."""
    n_global = len(le_global.classes_)
    gate_probs  = gate_model.predict_proba(X_mfcc_scaled)   # (N, 3)
    pred_states = gate_probs.argmax(axis=1)
    state_names = le_state.inverse_transform(pred_states)

    out = np.zeros((len(X_mfcc_scaled), n_global), dtype=np.float32)
    for i, state in enumerate(state_names):
        sub_probs = expert_models[state].predict_proba(X_mfcc_scaled[i:i+1])[0]
        for local_idx, subclass in enumerate(le_experts[state].classes_):
            global_idx = le_global.transform([subclass])[0]
            out[i, global_idx] = sub_probs[local_idx]
    return out

print('Computing soft probabilities for tie-breaking...')
proba_moe_panns = moe_panns_proba(X_panns_test, gate_model, expert_models,
                                   le_state, le_experts, le)
proba_moe_xgb   = moe_xgb_proba(X_mfcc_test_s, gate_xgb, expert_xgb_models,
                                  le_state, le_experts, le)
print('Done.')


Computing soft probabilities for tie-breaking...
Done.


In [91]:
# ── Hard majority vote with soft tie-breaking ─────────────────────────
# Sum of probabilities across the three voters (used only when votes split 1-1-1)
proba_sum = proba_xgb + proba_moe_panns + proba_moe_xgb   # (N, 9)

y_pred_vote = np.empty(len(all_votes), dtype=int)

for i in range(len(all_votes)):
    row      = all_votes[i]                # [v1, v2, v3]
    majority = stats.mode(row, keepdims=True).mode[0]  # most common class index
    count    = np.sum(row == majority)

    if count >= 2:
        # Clear majority (2 or 3 voters agree)
        y_pred_vote[i] = majority
    else:
        # 3-way tie — pick class with highest summed probability
        y_pred_vote[i] = np.argmax(proba_sum[i])

# ── Evaluate ensemble ────────────────────────────────────────────────
acc_vote = accuracy_score(y_test, y_pred_vote)
f1_vote  = f1_score(y_test, y_pred_vote, average='weighted')

print('=' * 60)
print('  ENSEMBLE VOTE — TOP-3 MODELS')
print('=' * 60)
print(f'  Accuracy    : {acc_vote:.4f}')
print(f'  Weighted F1 : {f1_vote:.4f}')
print()
print(f'  (XGBoost alone         : {acc_xgb:.4f})')
print(f'   MoE PANNs alone        : {acc_hier:.4f})')
print(f'   MoE XGBoost alone      : {acc_xgb_hier:.4f})')
print('=' * 60)
print()
print(classification_report(y_test, y_pred_vote, target_names=le.classes_))


  ENSEMBLE VOTE — TOP-3 MODELS
  Accuracy    : 0.9155
  Weighted F1 : 0.9142

  (XGBoost alone         : 0.8851)
   MoE PANNs alone        : 0.8378)
   MoE XGBoost alone      : 0.8649)

                       precision    recall  f1-score   support

         bad_ignition       0.96      0.86      0.91        28
         dead_battery       0.96      1.00      0.98        26
              low_oil       0.82      0.72      0.77        32
        normal_brakes       0.85      1.00      0.92        35
   normal_engine_idle       0.93      1.00      0.96        40
normal_engine_startup       0.87      0.96      0.91        27
       power_steering       0.92      0.87      0.89        39
      serpentine_belt       1.00      0.94      0.97        35
      worn_out_brakes       0.94      0.88      0.91        34

             accuracy                           0.92       296
            macro avg       0.92      0.92      0.91       296
         weighted avg       0.92      0.92      0.91    

In [ ]:
# ── Confusion matrix for the ensemble ────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
plot_confusion_matrix(
    y_test, y_pred_vote, le.classes_,
    f'Ensemble Vote (Top-3 Models)\nAcc={acc_vote:.4f} | F1={f1_vote:.4f}',
    ax
)
plt.tight_layout()
plt.show()


In [93]:
# ── Agreement analysis — how often do the three models agree? ────────
agree_all   = np.sum(np.all(all_votes == all_votes[:, [0]], axis=1))   # unanimous
agree_two   = np.sum(np.any(                                           # exactly 2 agree
    np.stack([
        all_votes[:, 0] == all_votes[:, 1],
        all_votes[:, 0] == all_votes[:, 2],
        all_votes[:, 1] == all_votes[:, 2],
    ], axis=1), axis=1)) - agree_all
disagree_all = len(all_votes) - agree_all - agree_two

print(f'Vote agreement breakdown (N={len(all_votes)}):')
print(f'  All 3 agree    : {agree_all:4d}  ({agree_all/len(all_votes):.1%})')
print(f'  2-of-3 agree   : {agree_two:4d}  ({agree_two/len(all_votes):.1%})')
print(f'  3-way tie      : {disagree_all:4d}  ({disagree_all/len(all_votes):.1%})')

# Accuracy when all agree vs when they split
mask_all   = np.all(all_votes == all_votes[:, [0]], axis=1)
mask_split = ~mask_all
if mask_all.sum():
    print(f'\n  Accuracy (unanimous)  : {accuracy_score(y_test[mask_all],  y_pred_vote[mask_all]):.4f}')
if mask_split.sum():
    print(f'  Accuracy (split vote) : {accuracy_score(y_test[mask_split], y_pred_vote[mask_split]):.4f}')


Vote agreement breakdown (N=296):
  All 3 agree    :  215  (72.6%)
  2-of-3 agree   :   66  (22.3%)
  3-way tie      :   15  (5.1%)

  Accuracy (unanimous)  : 0.9953
  Accuracy (split vote) : 0.7037


## Inference — Predict a New Audio File

In [ ]:
def predict_audio(wav_path: str, model_choice: str = 'cnn'):
    """
    Predict the class of a new audio file.
    model_choice: 'xgboost' | 'cnn' | 'cnn_lstm' | 'yamnet'
    """
    wav_path = str(wav_path)

    if model_choice == 'xgboost':
        feat = extract_mfcc_features(wav_path).reshape(1, -1)
        feat = scaler.transform(feat)
        probs = xgb_model.predict_proba(feat)[0]

    elif model_choice == 'cnn':
        mel = extract_mel_spectrogram_array(wav_path, target_length=MEL_TIME_DIM)
        mel = mel[np.newaxis, ..., np.newaxis]
        probs = cnn_model.predict(mel, verbose=0)[0]

    elif model_choice == 'cnn_lstm':
        mel = extract_mel_spectrogram_array(wav_path, target_length=MEL_TIME_DIM)
        mel = mel[np.newaxis, ..., np.newaxis]
        probs = cnn_lstm_model.predict(mel, verbose=0)[0]

    elif model_choice == 'yamnet':
        y = extract_yamnet_waveform(wav_path)
        _, emb, _ = yamnet_model(y)
        emb_mean = emb.numpy().mean(axis=0).reshape(1, -1)
        probs = yamnet_head.predict(emb_mean, verbose=0)[0]

    else:
        raise ValueError(f'Unknown model: {model_choice}')

    pred_idx   = np.argmax(probs)
    pred_label = le.classes_[pred_idx]
    confidence = probs[pred_idx]

    print(f'Predicted class : {pred_label}')
    print(f'Confidence      : {confidence:.2%}')
    print(f'Parent state    : {STATE_MAP.get(pred_label, "unknown")}')

    # Bar chart of class probabilities
    fig, ax = plt.subplots(figsize=(9, 3))
    colors = [STATE_COLORS.get(STATE_MAP.get(c, ''), '#888') for c in le.classes_]
    bars = ax.barh(le.classes_, probs, color=colors, edgecolor='white')
    ax.axvline(x=max(probs), color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'Prediction Probabilities — {model_choice.upper()}', fontweight='bold')
    ax.set_xlabel('Probability')
    for bar, p in zip(bars, probs):
        if p > 0.01:
            ax.text(p + 0.005, bar.get_y() + bar.get_height()/2,
                    f'{p:.2%}', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
    return pred_label, confidence

# Example: predict a random test file
test_file = df.iloc[idx_test[0]]['path']
true_label = df.iloc[idx_test[0]]['label']
print(f'True label: {true_label}\n')
predict_audio(test_file, model_choice='cnn')

## Results Summary

In [87]:
# Per-class F1 for each model
def per_class_f1(y_true, y_pred, classes):
    report = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    return {cls: report[cls]['f1-score'] for cls in classes}

models_info = [
    ('XGBoost (MFCCs)',            acc_xgb,      f1_xgb,      y_pred_xgb),
    ('CNN (Mel Spectrogram)',      acc_cnn,      f1_cnn,      y_pred_cnn),
    ('CNN-LSTM Hybrid',            acc_cnn_lstm, f1_cnn_lstm, y_pred_cnn_lstm),
    ('YAMNet Transfer Learning',   acc_yamnet,   f1_yamnet,   y_pred_yamnet),
    ('PANNs CNN14',                acc_panns,    f1_panns,    y_pred_panns),
    ('Mixture-of-Experts (PANNs)', acc_hier,     f1_hier,     y_pred_fine_enc),
    ('Mixture-of-Experts (XGB)',   acc_xgb_hier, f1_xgb_hier, y_pred_fine_xgb_enc)    
]

# Summary table
summary_rows = []
for name, acc, f1, y_pred in models_info:
    pcf1 = per_class_f1(y_test, y_pred, le.classes_)
    row  = {'Model': name, 'Accuracy': round(acc, 4), 'Weighted F1': round(f1, 4)}
    row.update({f'F1 [{c}]': round(v, 3) for c, v in pcf1.items()})
    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows).set_index('Model')

print('=' * 70)
print('MODEL COMPARISON — TEST SET')
print('=' * 70)
display_cols = ['Accuracy', 'Weighted F1']
print(results_df[display_cols].to_string())
print()
print('Best model by Accuracy:    ', results_df['Accuracy'].idxmax())
print('Best model by Weighted F1: ', results_df['Weighted F1'].idxmax())

MODEL COMPARISON — TEST SET
                            Accuracy  Weighted F1
Model                                            
XGBoost (MFCCs)               0.8851       0.8827
CNN (Mel Spectrogram)         0.0811       0.0391
CNN-LSTM Hybrid               0.1453       0.0803
YAMNet Transfer Learning      0.7905       0.7904
PANNs CNN14                   0.8615       0.8614
Mixture-of-Experts (PANNs)    0.8378       0.8388
Mixture-of-Experts (XGB)      0.8649       0.8648

Best model by Accuracy:     XGBoost (MFCCs)
Best model by Weighted F1:  XGBoost (MFCCs)


In [ ]:
# Bar chart: Accuracy & F1 comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Performance Comparison — Test Set', fontsize=14, fontweight='bold', y=1.01)

model_names  = [m[0] for m in models_info]
accuracies   = [m[1] for m in models_info]
f1_scores_   = [m[2] for m in models_info]
bar_colors   = ['#E07B54', '#5B8DB8', '#9B7FB6', '#6BAE75']

for ax, values, metric in zip(axes, [accuracies, f1_scores_], ['Accuracy', 'Weighted F1']):
    bars = ax.barh(model_names, values, color=bar_colors, edgecolor='white', height=0.55)
    ax.set_xlim(0, 1.08)
    ax.set_xlabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.axvline(x=max(values), color='gray', linestyle='--', alpha=0.5, linewidth=1)
    for bar, val in zip(bars, values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 heatmap across all models
pcf1_data = {}
for name, acc, f1, y_pred in models_info:
    pcf1_data[name] = per_class_f1(y_test, y_pred, le.classes_)

pcf1_df = pd.DataFrame(pcf1_data).T   # models as rows, classes as columns

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    pcf1_df, annot=True, fmt='.2f', cmap='YlGn',
    linewidths=0.5, linecolor='white', vmin=0, vmax=1, ax=ax
)
ax.set_title('Per-Class F1 Score — All Models', fontsize=13, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Model')
ax.tick_params(axis='x', rotation=40, labelsize=9)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Training curves overlay (deep learning models only)
dl_histories = [
    (history_cnn,      'CNN',          '#5B8DB8'),
    (history_cnn_lstm, 'CNN-LSTM',     '#9B7FB6'),
    (history_yamnet,   'YAMNet Head',  '#6BAE75'),
    (history_panns,    'PANNs CNN14',  '#E07B54'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves — Deep Learning Models', fontsize=13, fontweight='bold')

for hist, name, color in dl_histories:
    ep = range(1, len(hist.history['accuracy']) + 1)
    axes[0].plot(ep, hist.history['val_accuracy'], label=name, color=color, linewidth=2)
    axes[1].plot(ep, hist.history['val_loss'],     label=name, color=color, linewidth=2)

axes[0].set_title('Validation Accuracy', fontweight='bold')
axes[0].set_ylabel('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].set_title('Validation Loss', fontweight='bold')
axes[1].set_ylabel('Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# All 6 confusion matrices side-by-side
fig, axes = plt.subplots(3, 2, figsize=(18, 21))
fig.suptitle('Confusion Matrices — All Models (Normalised)', fontsize=15, fontweight='bold', y=1.01)

for ax, (name, acc, f1, y_pred) in zip(axes.flat, models_info):
    plot_confusion_matrix(y_test, y_pred, le.classes_, f'{name}\nAcc={acc:.4f} | F1={f1:.4f}', ax)

plt.tight_layout()
plt.show()

# Results & Discussion

In [59]:
print('═' * 65)
print('  FINAL RESULTS SUMMARY')
print('═' * 65)
print(f'{"Model":<30} {"Accuracy":>10} {"Weighted F1":>13}')
print('─' * 55)
for name, acc, f1, _ in sorted(models_info, key=lambda x: -x[1]):
    marker = ' ← best' if acc == max(m[1] for m in models_info) else ''
    print(f'{name:<30} {acc:>10.4f} {f1:>13.4f}{marker}')
print('═' * 65)
print()
print('Recommendations:')
print('  • Best overall accuracy  →', results_df['Accuracy'].idxmax())
print('  • Best weighted F1       →', results_df['Weighted F1'].idxmax())
print('  • Fastest to train       → XGBoost (MFCCs)')
print('  • Most interpretable     → XGBoost (MFCCs) — check feature_importances_')
print('  • Best for small dataset → YAMNet Transfer Learning')
print()
print('XGBoost top-10 MFCC feature importances:')
feat_imp = pd.Series(xgb_model.feature_importances_).nlargest(10)
print(feat_imp.to_string())

═════════════════════════════════════════════════════════════════
  FINAL RESULTS SUMMARY
═════════════════════════════════════════════════════════════════
Model                            Accuracy   Weighted F1
───────────────────────────────────────────────────────
XGBoost (MFCCs)                    0.8851        0.8827 ← best
PANNs CNN14                        0.8615        0.8614
Mixture-of-Experts                 0.8311        0.8303
YAMNet Transfer Learning           0.7905        0.7904
CNN-LSTM Hybrid                    0.1453        0.0803
CNN (Mel Spectrogram)              0.0811        0.0391
═════════════════════════════════════════════════════════════════

Recommendations:
  • Best overall accuracy  → XGBoost (MFCCs)
  • Best weighted F1       → XGBoost (MFCCs)
  • Fastest to train       → XGBoost (MFCCs)
  • Most interpretable     → XGBoost (MFCCs) — check feature_importances_
  • Best for small dataset → YAMNet Transfer Learning

XGBoost top-10 MFCC feature importances:


# Conclusion

# Recommendation

# References

1. Kaggle Car Diagnostics Dataset - https://www.kaggle.com/datasets/malakragaie/car-diagnostics-dataset
2. Free car sounrds - https://pixabay.com/sound-effects/search/car-engine/